# Corrigé · Exercice 2.1 : un ticket P1 exige une approbation humaine

> Notebook autonome. Points clés : **aucun effet de bord avant `interrupt()`**, création dans un nœud séparé, clé d'idempotence = `thread_id`.

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

In [ ]:
from typing import Literal, TypedDict

from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

from helpdesk_ops.donnees import CATEGORIES
from helpdesk_ops.ticketing import Ticketing
from helpdesk_ops.tri import creer_classifieur, estimer_priorite, file_attente, normaliser_categorie

classifieur = creer_classifieur(llm)
ticketing = Ticketing()   # outil de ticketing simulé, vide au départ


class EtatTicket(TypedDict, total=False):
    ticket: str            # le texte de la demande
    categorie: str
    priorite: str          # P1 ou P3 (règle de J1)
    validation: dict       # la décision humaine : {"action", "par", "motif"?}
    ticket_id: str


def qualifier(state: EtatTicket) -> dict:
    """Fourni : catégorie par le LLM (J1), priorité par la règle explicable (défi J1-01)."""
    brute = classifieur.invoke({"ticket": state["ticket"], "categories": ", ".join(CATEGORIES)})
    return {"categorie": normaliser_categorie(brute), "priorite": estimer_priorite(state["ticket"])}

In [ ]:
def router_priorite(state: EtatTicket) -> Literal["validation_humaine", "creer_ticket"]:
    return "validation_humaine" if state["priorite"] == "P1" else "creer_ticket"


def validation_humaine(state: EtatTicket) -> Command[Literal["creer_ticket", "ticket_refuse"]]:
    # Rien avant interrupt() : ce code est rejoué à la reprise
    reponse = interrupt({
        "type": "validation_ticket_p1",
        "ticket_propose": {"demande": state["ticket"], "categorie": state["categorie"], "priorite": state["priorite"]},
        "actions": ["approuver", "rejeter"],
    })
    if not isinstance(reponse, dict) or reponse.get("action") not in ("approuver", "rejeter") or not reponse.get("par"):
        # Réponse invalide : on refuse par prudence (variante : reposer la question, cf. J2-02 section 5)
        return Command(update={"validation": {"action": "rejeter", "par": "systeme", "motif": f"réponse invalide {reponse!r}"}},
                       goto="ticket_refuse")
    return Command(update={"validation": reponse},
                   goto="creer_ticket" if reponse["action"] == "approuver" else "ticket_refuse")


def creer_ticket(state: EtatTicket, config: RunnableConfig) -> dict:
    ticket = ticketing.creer(
        cle_idempotence=config["configurable"]["thread_id"],   # une demande = un thread = au plus un ticket
        categorie=state["categorie"], priorite=state["priorite"], resume=state["ticket"][:80],
        file=file_attente(state["categorie"]), valide_par=(state.get("validation") or {}).get("par"),
    )
    return {"ticket_id": ticket.id}


def ticket_refuse(state: EtatTicket) -> dict:
    return {"ticket_id": None}


b = StateGraph(EtatTicket)
b.add_node("qualifier", qualifier)
b.add_node("validation_humaine", validation_humaine)
b.add_node("creer_ticket", creer_ticket)
b.add_node("ticket_refuse", ticket_refuse)
b.add_edge(START, "qualifier")
b.add_conditional_edges("qualifier", router_priorite)
b.add_edge("creer_ticket", END)
b.add_edge("ticket_refuse", END)
circuit = b.compile(checkpointer=InMemorySaver())

from IPython.display import Markdown, display
display(Markdown(f"```mermaid\n{circuit.get_graph().draw_mermaid()}\n```"))

In [ ]:
DEMANDES = {
    "a": "URGENT : l'ERP de production est bloqué pour toute la compta",   # P1 → approuvé
    "b": "Production arrêtée : plus aucun scanner ne fonctionne à l'entrepôt",  # P1 → rejeté
    "c": "Je voudrais un deuxième écran pour mon poste",                     # P3 → créé directement
}

INITIALES = "corrige"


def config_de(lettre: str) -> dict:
    thread = f"ex21-{INITIALES}-{lettre}"
    return {"configurable": {"thread_id": thread}, "callbacks": [langfuse_handler], "run_name": "ex21"}


def lancer(lettre: str, entree) -> None:
    with propagate_attributes(session_id=config_de(lettre)["configurable"]["thread_id"], tags=["jour-2", "ex21"]):
        s = circuit.invoke(entree, config_de(lettre), version="v2")
    attente = s.interrupts[0].value["ticket_propose"] if s.interrupts else None
    print(f"[{lettre}] priorité={s.value.get('priorite')} | en attente={attente is not None} | ticket={s.value.get('ticket_id')}")


for lettre, texte in DEMANDES.items():
    lancer(lettre, {"ticket": texte})

print("\n--- décisions du responsable ---")
lancer("a", Command(resume={"action": "approuver", "par": "resp-n2"}))
lancer("b", Command(resume={"action": "rejeter", "par": "resp-n2", "motif": "incident déjà ouvert (NT-0042)"}))

print("\n--- seconde reprise de « a » : le thread est terminé, rien ne repart ---")
lancer("a", Command(resume={"action": "approuver", "par": "resp-n2"}))

print("\n--- rejeu de creer_ticket depuis son checkpoint (panne simulée après la création) ---")
avant = [h for h in circuit.get_state_history(config_de("a")) if h.next == ("creer_ticket",)][-1]
with propagate_attributes(session_id=config_de("a")["configurable"]["thread_id"], tags=["jour-2", "ex21"]):
    rejeu = circuit.invoke(None, {**avant.config, "callbacks": [langfuse_handler], "run_name": "ex21-rejeu"})
print(f"[a] rejeu : ticket={rejeu['ticket_id']} (le même : la clé d'idempotence a joué)")

print("\nTicketing :")
for t in ticketing.lister():
    print(f"  {t.id} {t.priorite} {t.categorie:<9} validé par {t.valide_par or '-':<8} ← {t.resume}")
langfuse.flush()

## Ce qu'il fallait observer

- **a** (P1) s'arrête, puis est créé après approbation avec `valide_par = resp-n2` ; **b** (P1) s'arrête puis est refusé : **rien** dans le ticketing ; **c** (P3) est créé directement, sans validation.
- **Seconde reprise** : elle ne prouve rien. Le thread est terminé, aucun nœud ne repart, que la clé d'idempotence soit là ou non.
- **Rejeu depuis le checkpoint** : `creer_ticket` est réellement réexécuté (visible dans la trace `ex21-rejeu`) et renvoie le **même** `ticket_id` : c'est la clé d'idempotence qui évite le doublon. Contre-épreuve avec une clé `uuid4()` : un second ticket apparaît (vérifié le 30/10/2026, LangGraph 1.2).
- **Langfuse** : chaque demande P1 donne **deux** traces dans sa session (avant et après la validation). La décision humaine est dans l'état et dans le ticket (`valide_par`), pas dans la trace : pour un audit (NIS2), il faut aussi l'horodatage et l'identité authentifiée du validateur, pas un simple texte.

## Pour aller plus loin

- Ajoutez l'action « modifier » (P1 → P2) : le ticket doit être créé en P2 **sans** nouvelle validation.
- Que se passe-t-il si personne ne répond pendant 3 jours ? Proposez une règle (relance, escalade, expiration) et où la coder.